# Test — `cruxes` : Drawing Trajectories for Body Movements

Banc d'essai de [tommyjtl/climbing-analysis-toolbox](https://github.com/tommyjtl/climbing-analysis-toolbox) (paquet `cruxes`) sur `example/example-2.mp4`.

**Totalement isolé du projet** : `cruxes` épingle `mediapipe==0.10.32` + `torch` + `image-matching-models`, incompatible avec le venv `apps/ai`. Il est donc installé dans son propre env via `uv tool` — rien n'est ajouté au `pyproject.toml`, et ce notebook n'a aucun rapport avec `src/pipelines/`.

## 1. Installation (env isolé)

In [ ]:
!uv tool install --python 3.11 --torch-backend=cpu "git+https://github.com/tommyjtl/climbing-analysis-toolbox"
!cruxes body-trajectory --help

## 2. `body-trajectory` sur example-2.mp4

Points suivis : bassin, centre haut du corps, tête, mains, pieds. Historique 2 s, lissage gaussien, squelette + jauges de vitesse.

⚠️ **Patch nécessaire ici** : `cruxes` réutilise le FOURCC de la vidéo source ([`body_trajectory.py:1319`](https://github.com/tommyjtl/climbing-analysis-toolbox/blob/main/src/cruxes/utils/body_trajectory.py#L1319)). `example-2.mp4` est en H.264, et l'OpenCV embarqué n'a pas d'encodeur H.264 utilisable sur cette machine (il tombe sur `h264_v4l2m2m` → échec silencieux, JSON écrits mais **aucune vidéo**). On force donc `mp4v` sur le `VideoWriter` en passant par l'API Python plutôt que par la CLI.

In [ ]:
%%bash
unset MPLBACKEND  # Jupyter injects matplotlib_inline's backend, unknown to cruxes' own matplotlib
~/.local/share/uv/tools/cruxes/bin/python - <<'PY'
import cv2

_VW = cv2.VideoWriter
cv2.VideoWriter = lambda path, fourcc, *a: _VW(path, cv2.VideoWriter_fourcc(*"mp4v"), *a)

from cruxes import Cruxes

Cruxes().body_trajectory(
    "example/example-2.mp4",
    track_point=["hip_mid", "upper_body_center", "head",
                 "left_hand", "right_hand", "left_foot", "right_foot"],
    draw_pose=True,
    show_trajectory=True,
    show_gauges=True,
    trajectory_history_seconds=2.0,
    smoothing="gaussian",
    use_cached_landmarks=True,
    export_landmarks=True,
    export_metadata=True,
)
PY

## 3. Résultat

La sortie est en MPEG-4 part 2, illisible dans un navigateur → transcodage H.264 pour l'affichage.

In [ ]:
from IPython.display import Video

!ffmpeg -v error -y -i example/pose_trajectory_example-2.mp4 \
    -c:v libx264 -pix_fmt yuv420p -crf 23 example/pose_trajectory_example-2-h264.mp4

Video("example/pose_trajectory_example-2-h264.mp4", embed=True, width=420)

## 4. Artefacts produits

| Fichier | Contenu |
|---|---|
| `example/pose_trajectory_example-2.mp4` | Vidéo annotée : squelette blanc, trajectoires colorées par vitesse, flèches de vélocité, jauges |
| `example/example-2_landmarks.json` | 33 landmarks MediaPipe × 309 frames (cache réutilisable via `use_cached_landmarks`) |
| `example/example-2_trajectory_metadata.json` | Trajectoires + vitesses par point suivi, format destiné au front |

In [ ]:
import json

meta = json.load(open("example/example-2_trajectory_metadata.json"))
print(json.dumps(meta, indent=1)[:1200])